# 📚 Week 6 · Ask the course: answers from *your* documents

Last week's app answered from what the model **remembered**. Today's app answers from **the course notes**: it finds the right slides first, reads them, and answers with the slide it used. When the notes don't say, it says so.

That technique has a name, and it's this week's easter egg 🥚.

We build it in four layers: a no-code version → paste every note into the prompt → find the right slides first → a grounded answer in a small app. Then we **test it with five questions and score it**, exactly as you'll do for your capstone.

**How to use this notebook**
- Run each cell with ▶ (or **Shift + Enter**), top to bottom.
- Cells marked **✏️ YOUR TURN** are where you change one line and run again.
- Stuck? Type the **section number** in the Zoom chat.

**Before you start:** File → **Save a copy in Drive**, so your changes are yours. Same key as Week 5; nothing new to set up.

## 🏠 Start here · The problem

Every slide from Weeks 1–5 is one row of a spreadsheet: **week, slide, title, text**. That's about 250 slides, roughly a 100-page textbook. **[Open it as a table](https://github.com/san81/ai-course/blob/main/docs/weeks/week6/course-notes.csv)** and scroll for a minute.

You know these notes. That's the point: **you can tell when the AI gets them wrong.** Look out for:

- 🔁 **The same idea in several weeks:** temperature appears in Week 3, again in Week 4 and again in Week 5. Which one should it quote?
- 🔄 **Something that changed:** Week 4's slides say *Gemini Gems*. A Week 5 update says Gems are becoming *skills*.
- 🕳️ **Questions the notes can't answer:** the final exam date isn't anywhere. But the word "exam" is on lots of slides (*"studying for the exam"*), which makes it a trap.

**Our app:** a question goes in, and out comes **an answer from the notes, with the slide numbers it used**, or *"That's not in the course notes."*

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 200)

# Every slide from Weeks 1-5, one row each. No API key needed for this step.
DATA_URL = "https://raw.githubusercontent.com/san81/ai-course/main/docs/weeks/week6/course-notes.csv"

# ✏️ Your own documents later? See section 7 at the bottom (a Google Sheet or a PDF).
df = pd.read_csv(DATA_URL)
print(len(df), "slides ·", f"{df['text'].str.len().sum():,}", "characters")
df

In [ ]:
# 📝 Five test questions, the way a student would really ask, with what a good answer looks like.
# Your capstone has five of these too. They're how you know whether it works.
TESTS = [
    ("Who wrote to Hardy in Cambridge in 1913?",
     "Ramanujan (Week 4)"),
    ("wat is a embeding",                                   # typos on purpose
     "Text turned into a list of numbers; similar meanings sit close together (Week 2, Week 3)"),
    ("What are the five parts of a good brief, and what does temperature change?",
     "WHO · WHY · WHAT · HOW · SHOW, and how boldly the next word is picked (both Week 3)"),
    ("Should I build my no-code capstone as a Gemini Gem?",
     "No: Gems are becoming Gemini skills, so build a skill (Week 5 update)"),
    ("When is the final exam for this course?",
     "Not in the notes, and it should SAY so"),
]
for n, (question, good) in enumerate(TESTS, 1):
    print(f"{n}. {question}\n   ✅ good answer: {good}\n")

✏️ **Your turn (30 seconds):** the plain chatbot has never seen our notes. How many of these five will it get right? Type **one number (0–5)** in the Zoom chat. We'll find out in section 1.

## 0 · Setup (2 minutes)

Same as Week 5: your **`GEMINI_API_KEY`** should already be in the 🔑 Secrets panel (left sidebar), with **Notebook access** switched on. It's a new notebook, so switch Notebook access on again if Colab asks.

No key yet? [aistudio.google.com](https://aistudio.google.com) → **Get API key** → **Create API key** → add it as a secret named `GEMINI_API_KEY`.

Run the three cells below until you see **✅ Connected**.

In [ ]:
!pip -q install -U google-genai gradio

In [ ]:
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except ImportError:                      # not running in Colab
    import os, getpass
    API_KEY = os.environ.get("GEMINI_API_KEY") or getpass.getpass("Paste your Gemini API key: ")
except Exception as error:
    print("🔑 Add GEMINI_API_KEY in the Secrets panel (key icon, left) and switch on 'Notebook access'.")
    raise error

client = genai.Client(api_key=API_KEY)

MODEL = "gemini-3.5-flash-lite"   # ✏️ free tier. Another free option: "gemini-3.6-flash"
print("Key loaded. Now run the next cell to test it.")

In [ ]:
# 🛟 Turns confusing error messages into plain advice, then checks your key works.
# Run it; no need to read it.
import time
from google.genai import errors

def explain_api_error(e):
    code, text = getattr(e, "code", None), str(e)
    if code == 400 and "API key" in text:
        return ("🔑 Your key was not accepted. Copy it again from aistudio.google.com (no spaces), "
                "update the GEMINI_API_KEY secret, and run the setup cells again.")
    if code == 402:
        return ("💳 This key belongs to a project that is linked to billing, and that billing has no credit. "
                "You do NOT need billing or a card for this class. In aistudio.google.com create a NEW project "
                "(no billing), create a key in it, put that key in the GEMINI_API_KEY secret, and run the setup "
                "cells again. Never add a card just for this class.")
    if code == 403:
        return ("🚫 This Google account can't use the Gemini API. College accounts often block it: "
                "sign in with a personal Gmail, make a new key, and update the secret.")
    if code == 404:
        return ("🔍 That model name isn't available to your key. In the setup cell, change MODEL "
                "(for example to 'gemini-3.6-flash') and run it again.")
    if code == 429:
        return ("⏳ You've reached the free limit. If it's the per-minute limit, wait one minute and try again. "
                "If it's the daily limit, it resets at midnight US Pacific time (about 12:30–1:30 pm in India). "
                "Meanwhile, follow along on the instructor's screen.")
    if code and code >= 500:
        return "🌩️ Google's servers are busy. Wait 30 seconds and run the cell again."
    return None

def _friendly_errors(shell, etype, evalue, tb, tb_offset=None):
    advice = explain_api_error(evalue)
    if advice:
        print(advice)
        print(f"\n(Technical detail, if your instructor asks: error {getattr(evalue, 'code', '?')})")
    else:
        shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)

try:
    get_ipython().set_custom_exc((errors.APIError,), _friendly_errors)
except NameError:      # not running in a notebook
    pass

def with_retry(fn, tries=3):
    """Run fn(); if the free per-minute limit is hit, wait and try again."""
    for attempt in range(tries):
        try:
            return fn()
        except errors.APIError as e:
            if e.code == 429 and attempt < tries - 1:
                print("   ⏳ rate limit, waiting 30 s…")
                time.sleep(30)
            else:
                raise

# One tiny test call, so any key problem shows up now, not halfway through the class.
try:
    client.models.generate_content(model=MODEL, contents="Reply with just: OK")
    print("✅ Connected. Model:", MODEL)
except errors.APIError as e:
    print(explain_api_error(e) or e)

## 1 · Closed book: ask without the notes

First, the five questions with **no notes at all**: the model answers from memory, like a student in a closed-book exam. Watch for answers that sound sure but aren't from *our* course.

In [ ]:
# Rough prices for paid use, in US dollars per 1 million tokens (free tier costs ₹0).
PRICE_IN, PRICE_OUT, USD_TO_INR = 0.30, 2.50, 88     # gemini-3.5-flash-lite, Oct 2026

def tokens_and_cost(response):
    u = response.usage_metadata
    if u is None:
        return 0, 0, 0.0
    tin = u.prompt_token_count or 0
    tout = (u.candidates_token_count or 0) + (getattr(u, "thoughts_token_count", 0) or 0)
    rupees = (tin * PRICE_IN + tout * PRICE_OUT) / 1_000_000 * USD_TO_INR
    return tin, tout, rupees

CLOSED_BOOK = "You are a helpful assistant for students of an 8-week AI course. Answer in at most 60 words."

for n, (question, good) in enumerate(TESTS, 1):
    response = with_retry(lambda: client.models.generate_content(
        model=MODEL, contents=question,
        config=types.GenerateContentConfig(system_instruction=CLOSED_BOOK, temperature=0.2)))
    print(f"{n}. {question}\n   🤖 {response.text.strip()}\n   ✅ good answer: {good}\n")
    time.sleep(4)        # stay inside the free per-minute limit

✏️ **Your turn:** how many did it *really* get right, by our course's answers? Type **one number (0–5)**. Was it close to your guess?

Look especially at **4** (it doesn't know about the update) and **5** (does it invent a date, or admit it doesn't know?).

## Layer 1 · No code: hand it the notes (watch first)

The simplest open-book version needs no code:

1. Download the notes: open **[the table](https://github.com/san81/ai-course/blob/main/docs/weeks/week6/course-notes.csv)** → the **⬇ Download raw file** button (top right).
2. Open **[notebooklm.google.com](https://notebooklm.google.com)** (you used it in Week 4) → **New notebook** → upload the file as a source.
3. Ask question 4: *"Should I build my no-code capstone as a Gemini Gem?"*
4. Click the little number after the answer: it shows **the exact slide it used**.

*A Gemini skill can hold files too, and a Claude Project or Custom GPT takes uploads. Same idea, different app. 🧩 No-code track? This is your capstone path: your documents + your instructions.*

Everything below does the same thing **from code**, so you can see how it works and use it on documents too big to upload.

## Layer 2 · Paste *everything* into the prompt

The bluntest possible way: glue all 250 slides into one long text and send it **with every question**. Each slide gets a label like `[W3 · slide 24]` so the model can tell us where it found things.

In [ ]:
def source_of(row):
    """A short label for where a piece of text came from, e.g. 'W3 · slide 24'."""
    if "source" in row and pd.notna(row["source"]):
        return str(row["source"])
    if int(row["slide"]) == 0:
        return f"W{row['week']} · course update"
    return f"W{row['week']} · slide {row['slide']}"

ALL_NOTES = "\n\n".join(f"[{source_of(r)}] {r['title']}\n{r['text']}" for _, r in df.iterrows())

n_tokens = client.models.count_tokens(model=MODEL, contents=ALL_NOTES).total_tokens
print(f"📏 All the notes = {n_tokens:,} tokens")

OPEN_BOOK = """You answer students' questions about an 8-week AI course, using ONLY the course notes given.
Cite the labels you used, like [W3 · slide 24]. If the notes don't contain the answer, say:
"That's not in the course notes." Answer in at most 80 words."""

question = TESTS[3][0]       # ✏️ try another: TESTS[0][0] … TESTS[4][0]
response = with_retry(lambda: client.models.generate_content(
    model=MODEL,
    contents=f"COURSE NOTES:\n{ALL_NOTES}\n\nQUESTION: {question}",
    config=types.GenerateContentConfig(system_instruction=OPEN_BOOK, temperature=0.2)))

print("❓", question)
print("🤖", response.text.strip())
tin, tout, rupees = tokens_and_cost(response)
print(f"\n🔢 {tin:,} tokens in · {tout} tokens out · ≈ ₹{rupees:.3f} per question"
      f" · ≈ ₹{rupees * 1000:,.0f} per 1,000 questions")

✏️ **Your turn:** type the **tokens in** (just the number) in the chat.

**It works!** For a small set of notes, this is honestly the right answer, and it's what NotebookLM-style tools often do. So why not always?

- 💰 **Every question re-sends the whole book.** Look at the ₹ per 1,000 questions.
- 📚 **A 500-page textbook, or a whole year of notes, won't fit** (and the free tier limits how many tokens you can send per minute).
- 🌫️ **Long prompts blur.** A detail in the middle of a huge prompt is easier to miss than one in a short prompt.

So the next layer does what a good student does in an open-book exam: **find the right pages first, then read only those.**

## Layer 3 · Find the right slides first: chunk → embed → search

Three steps, and you already know the idea behind each one:

1. ✂️ **Chunk:** cut the notes into small pieces. Our slides are already pieces; long ones get cut again.
2. 🗺️ **Embed:** turn every piece into a list of numbers (an **embedding**, Week 2: *meaning is a place*). Pieces that mean similar things land close together.
3. 🔎 **Search:** turn the *question* into numbers the same way, and pick the pieces that sit closest to it.

The model isn't needed for steps 1–3. That's search (last week's egg: **information retrieval**).

In [ ]:
CHUNK_SIZE = 600     # ✏️ characters per piece. Try 300 or 1200 later (section 5).
OVERLAP = 100        # pieces share a little text, so a sentence cut in half still appears whole once

def chunk_text(text, size=CHUNK_SIZE, overlap=OVERLAP):
    if len(text) <= size:
        return [text]
    pieces, start = [], 0
    while start < len(text):
        end = min(start + size, len(text))
        if end < len(text):                       # try not to cut a word in half
            space = text.rfind(" ", start + size // 2, end)
            end = space if space != -1 else end
        pieces.append(text[start:end].strip())
        if end >= len(text):
            break
        start = max(end - overlap, start + 1)
    return pieces

chunks = []
for _, r in df.iterrows():
    for piece in chunk_text(str(r["text"])):
        chunks.append({"source": source_of(r), "title": str(r["title"]), "text": piece})

print(f"✂️ {len(df)} slides → {len(chunks)} pieces of at most {CHUNK_SIZE} characters")
pd.DataFrame(chunks).sample(5, random_state=1)

In [ ]:
import numpy as np

EMBED_MODEL = "gemini-embedding-001"   # Google's embedding model (turns text into numbers)
DIMENSIONS = 768                       # how many numbers per piece
EMBEDDER = "gemini"                    # switches to "local" by itself if the free limit runs out

_local_model = None
def _embed_local(texts):
    """Backup: a small free embedding model that runs inside Colab. No key, no limits."""
    global _local_model
    if _local_model is None:
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError:
            import subprocess, sys
            subprocess.run([sys.executable, "-m", "pip", "-q", "install", "sentence-transformers"], check=True)
            from sentence_transformers import SentenceTransformer
        _local_model = SentenceTransformer("all-MiniLM-L6-v2")
    return np.array(_local_model.encode(texts, normalize_embeddings=True))

def embed(texts, kind="document", titles=None):
    """Turn a list of texts into a table of numbers: one row (embedding) per text."""
    global EMBEDDER
    if EMBEDDER == "local":
        return _embed_local(texts)
    task = "RETRIEVAL_DOCUMENT" if kind == "document" else "RETRIEVAL_QUERY"
    rows = []
    try:
        for start in range(0, len(texts), 50):              # 50 pieces per call
            batch = texts[start:start + 50]
            result = with_retry(lambda: client.models.embed_content(
                model=EMBED_MODEL, contents=batch,
                config=types.EmbedContentConfig(task_type=task, output_dimensionality=DIMENSIONS)))
            rows += [e.values for e in result.embeddings]
            if len(texts) > 50:
                print(f"   🗺️ {min(start + 50, len(texts))} of {len(texts)} pieces")
                time.sleep(2)
    except errors.APIError as e:
        if e.code in (429, 404):
            print("⏳ The Gemini embedding limit was reached, so we switch to a free model that runs in Colab.\n"
                  "   Same idea, different numbers. Run the search cells again after this one.")
            EMBEDDER = "local"
            return _embed_local(texts)
        raise
    vectors = np.array(rows)
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)   # every row gets length 1

VECTORS = embed([c["title"] + ". " + c["text"] for c in chunks])
print(f"\n✅ {VECTORS.shape[0]} pieces, {VECTORS.shape[1]} numbers each ({EMBEDDER} embeddings)")
print("The first piece starts with these numbers:", np.round(VECTORS[0][:6], 3), "…")

In [ ]:
TOP_K = 4      # ✏️ how many pieces to hand to the model (try 2 or 8 in section 5)

def search(question, k=TOP_K):
    """The k pieces whose meaning sits closest to the question."""
    q = embed([question], kind="query")[0]
    if q.shape[0] != VECTORS.shape[1]:
        raise RuntimeError("🔁 The embeddings switched to the local model. Re-run the 🗺️ embedding cell above, then this one.")
    scores = VECTORS @ q                     # closeness: 1.0 = same meaning, 0 = unrelated
    best = np.argsort(-scores)[:k]
    return [dict(chunks[i], score=round(float(scores[i]), 3)) for i in best]

question = "wat is a embeding"      # ✏️ then try: "When is the final exam for this course?"
print("❓", question)
pd.DataFrame(search(question))[["score", "source", "title", "text"]]

✏️ **Your turn:** look at the **top row** of the table. Its `source` says something like *W2 · slide 19*. Type just that **week number** (1–5) in the chat. (Notice: the typos didn't matter. It searches by *meaning*, not by spelling.)

Then change the question to `"When is the final exam for this course?"` and run the cell again. That answer is **in none of our notes**, yet search still returns 4 slides, ones that only mention "exam" in passing. **Search always returns something.** It can't tell "close" from "the answer", so the next layer has to say *"That's not in the course notes."*

## Layer 4 · A grounded answer, then a small app

Now we put it together: **search → hand the top pieces to the model → answer only from them**, with the labels it used. This is the whole technique. In real products the instructions are almost exactly these.

In [ ]:
GROUNDED = """You answer students' questions about an 8-week AI course.
Use ONLY the notes below the question. Each note starts with a label like [W3 · slide 24].
- Answer in at most 80 words, simply, like a helpful senior student.
- After each fact, cite its label in square brackets.
- If two notes disagree, prefer the later week and say that it changed.
- If the notes don't contain the answer, reply exactly: "That's not in the course notes."
  Never guess, and never use what you remember from outside the notes."""

def ask_course(question, k=TOP_K):
    hits = search(question, k)
    notes = "\n\n".join(f"[{h['source']}] {h['title']}\n{h['text']}" for h in hits)
    response = with_retry(lambda: client.models.generate_content(
        model=MODEL,
        contents=f"QUESTION: {question}\n\nNOTES:\n{notes}",
        config=types.GenerateContentConfig(system_instruction=GROUNDED, temperature=0.2)))
    return response.text.strip(), hits, response

question = TESTS[3][0]       # ✏️ change me
answer, hits, response = ask_course(question)
print("❓", question)
print("🤖", answer)
print("\n📎 slides it was given:", " · ".join(h["source"] for h in hits))
tin, tout, rupees = tokens_and_cost(response)
print(f"🔢 {tin:,} tokens in (Layer 2 sent {n_tokens:,}) · ≈ ₹{rupees * 1000:,.1f} per 1,000 questions")

✏️ **Your turn:** compare the **tokens in** with Layer 2. Roughly how many times smaller? Type **one number** (for example *50*).

In [ ]:
import gradio as gr

def app(question):
    if not question.strip():
        return "Type a question first 🙂"
    answer, hits, _ = ask_course(question)
    used = "\n".join(f"- **{h['source']}** · {h['title']}" for h in hits)
    return f"{answer}\n\n---\n📎 *Slides it read:*\n{used}"

demo = gr.Interface(
    fn=app,
    inputs=gr.Textbox(lines=2, label="Ask anything about Weeks 1–5"),
    outputs=gr.Markdown(label="Answer from the notes"),
    title="📚 Ask the course",
    examples=[[q] for q, _ in TESTS],
)
demo.launch(share=True)

## 5 · Test it: five questions, scored

A demo that answers one question well proves nothing. **Run all five test questions, and mark each one yourself.** This is exactly what you'll do for your capstone.

In [ ]:
for n, (question, good) in enumerate(TESTS, 1):
    answer, hits, _ = ask_course(question)
    print(f"{n}. ❓ {question}\n   🤖 {answer}\n   📎 {' · '.join(h['source'] for h in hits)}\n   ✅ good answer: {good}\n")
    time.sleep(4)

In [ ]:
# ✏️ YOUR TURN: one letter per test, in order. Y = right, N = wrong.
MY_MARKS = "?????"

score = MY_MARKS.upper().count("Y")
print(f"🎯 Score: {score} of {len(TESTS)}")
if "?" in MY_MARKS:
    print("   (replace the ? marks with Y or N first)")

✏️ **Type your score out of 5** in the chat.

**When one is wrong, find out *where* it went wrong.** Look at the 📎 slides it was given:

| What you see | Where it broke | What to try |
|---|---|---|
| The right slide is **not** in the 📎 list | 🔎 **Search** found the wrong pieces | change `CHUNK_SIZE`, raise `TOP_K`, or reword the question |
| The right slide **is** there, but the answer is wrong | ✍️ **The answer** ignored it | add a line to `GROUNDED` |
| The notes don't have it, and it answered anyway | 🕳️ **It guessed** | make the "not in the notes" rule stronger |

✏️ **Fix one:** change *one* thing (for example `CHUNK_SIZE = 300` or `TOP_K = 8`), re-run from the ✂️ chunk cell down, and score again. **Change one thing at a time**, or you won't know which change helped.

*Grown-up versions of this table: tools like **promptfoo** and **Ragas** run a test set like yours automatically after every change.*

## 6 · Make it real: a vector database

Our `VECTORS` table lives in memory and disappears when Colab closes. With thousands of documents, you'd also want search to stay fast. A **vector database** stores the pieces and their numbers, and does the "find the closest" step for you. **Chroma** is a free one that runs right here.

Same pieces, same numbers. Only the storage changes.

In [ ]:
!pip -q install chromadb

In [ ]:
import chromadb

db = chromadb.Client()      # in memory. To keep it: chromadb.PersistentClient(path="/content/drive/MyDrive/course-db")
collection = db.get_or_create_collection("course_notes")
collection.add(
    ids=[str(i) for i in range(len(chunks))],
    embeddings=VECTORS.tolist(),
    documents=[c["text"] for c in chunks],
    metadatas=[{"source": c["source"], "title": c["title"]} for c in chunks],
)
print("🗄️ Pieces stored in Chroma:", collection.count())

question = "Why does the same prompt give different answers?"     # ✏️ change me
found = collection.query(query_embeddings=embed([question], kind="query").tolist(), n_results=TOP_K)
print("❓", question)
for meta, text in zip(found["metadatas"][0], found["documents"][0]):
    print(f"   📎 {meta['source']} · {meta['title']}\n      {text[:110]}…")

### Optional · keyword search (BM25) and hybrid search

Meaning search isn't the only kind. **BM25** is the classic *keyword* search: rare words that match count most. Run both on the same questions and compare: keywords win on rare exact names (`MYCIN`), meaning wins on typos and new wording (`wat is a embeding`). **Hybrid search** merges the two lists.

In [ ]:
!pip -q install rank_bm25

In [ ]:
import re
from rank_bm25 import BM25Okapi

STOP = {"a", "an", "the", "is", "are", "was", "what", "when", "who", "how", "why", "do", "does", "did",
        "of", "to", "in", "for", "on", "and", "or", "it", "i", "my", "this", "that", "with", "be"}

def words(text):
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP]   # skip filler words

bm25 = BM25Okapi([words(c["title"] + " " + c["text"]) for c in chunks])

def keyword_search(question, k=TOP_K):
    scores = bm25.get_scores(words(question))
    best = [i for i in np.argsort(-scores)[:k] if scores[i] > 0]     # no matching word = no result
    return [dict(chunks[i], score=round(float(scores[i]), 2)) for i in best]

def hybrid_search(question, k=TOP_K):
    """Merge both lists: a piece near the top of EITHER list scores well (reciprocal rank fusion)."""
    merged = {}
    for results in (search(question, 20), keyword_search(question, 20)):
        for rank, hit in enumerate(results):
            key = (hit["source"], hit["text"][:50])
            merged.setdefault(key, [hit, 0.0])[1] += 1 / (60 + rank)
    return [hit for hit, _ in sorted(merged.values(), key=lambda x: -x[1])[:k]]

for question in ["MYCIN", "wat is a embeding", "When is the final exam?"]:    # ✏️ add your own
    print("❓", question)
    print("   🔤 keyword:", " · ".join(h["source"] for h in keyword_search(question)) or "nothing matched")
    print("   🗺️ meaning:", " · ".join(h["source"] for h in search(question)))
    print("   🔀 hybrid: ", " · ".join(h["source"] for h in hybrid_search(question)), "\n")

**When your data gets huge** (millions of pieces, many users at once), the same idea moves to a server: **Qdrant**, **OpenSearch** (keyword + meaning search in one system), **pgvector** (inside PostgreSQL) or a managed service like Pinecone. The tricks that keep it fast:
- ⚡ **approximate search** (HNSW): check a smart shortlist of neighbours, not every piece
- 🏷️ **filter first** by labels (`where={"source": ...}` in Chroma), then search by meaning
- 🗜️ **fewer numbers per piece**: `DIMENSIONS = 256` instead of 768, or compressed numbers
- 🔁 **embed once**, store the numbers, and re-embed only documents that changed

**Shortcuts you'll see online.** Libraries like **LlamaIndex** and **LangChain** wrap everything from Layer 3 to here into a few lines:

```python
# LlamaIndex (needs its own setup for Gemini; not run here)
documents = SimpleDirectoryReader("my_notes/").load_data()
index = VectorStoreIndex.from_documents(documents)       # chunk + embed + store
print(index.as_query_engine().query("What is temperature?"))   # search + grounded answer
```

Handy, and now you know what those three lines do underneath. When a shortcut gives a bad answer, you'll know where to look: the pieces, the search, or the instructions.

## 7 · Make it yours: your own documents

Two ways to swap the course notes for **your** documents (your syllabus, notes, a brochure, your capstone files):

**A · A Google Sheet** with columns `source`, `title`, `text` (one row per page or section) → File → Share → **Publish to web** → **Comma-separated values (.csv)** → paste the link as `DATA_URL` in the "Start here" cell.

**B · A PDF:** run the cell below and pick the file. Every page becomes one row.

Then re-run from **Layer 3 (✂️ chunk)** down, and **write five test questions about your documents** in `TESTS`.

In [ ]:
# Optional: turn a PDF into rows (one per page). Then re-run from the ✂️ chunk cell.
!pip -q install pypdf
from pypdf import PdfReader
from google.colab import files

uploaded = files.upload()                      # pick your PDF
name = next(iter(uploaded))
pages = PdfReader(name).pages
df = pd.DataFrame([{"source": f"{name} · p.{n}", "title": name, "text": p.extract_text() or ""}
                   for n, p in enumerate(pages, 1)])
df = df[df["text"].str.strip() != ""]
print(f"📄 {len(df)} pages with text. Now re-run from Layer 3 (✂️ chunk) and write your own TESTS.")
df.head()

---
### 🧠 RAG or fine-tuning?

- **RAG** (today) *adds facts*: the model reads your documents at the moment you ask. Update the documents and the answers update, with no training.
- **Fine-tuning** *changes behaviour*: style, format, tone, a narrow skill. It's training, so it costs time and data, and it's a poor way to teach facts that change.

Most capstones need RAG, a good prompt, or both. Picked a training project? The optional **deep-dive lab** after this week covers fine-tuning on a free Colab GPU.

**Next week:** the model stops only answering and starts *acting*: agents, tools and workflows. 🤖